# 16 - Executor-partition benchmark control

Runs in `PREPARE` or `EVALUATE` mode for notebook 15.

- `PREPARE` validates the benchmark array, stages every distinct source once, probes media metadata, verifies executor access, and writes run-scoped Delta input.
- `EVALUATE` checks terminal-result completeness, identity, resource snapshots, memory/thread safety, sustained duration, throughput, optional expected line counts, and an optional compatible notebook-04 baseline.

Use this notebook only through `pc-executor-partition-benchmark`. Notebook 15 must run once over the complete prepared batch; do not wrap it in a per-video ForEach.


In [ ]:
MODE = "PREPARE"  # PREPARE or EVALUATE
BENCHMARK_BATCH_ID = ""
BENCHMARK_ITEMS_JSON = ""
EXPECTED_ITEM_COUNT = 0
RUN_ID = ""

DATABASE = ""
TABLE_PREFIX = "people_counter"
INPUT_TABLE = ""
OUTPUT_TABLE = ""
RESOURCE_SNAPSHOT_TABLE = ""
PARTITION_PLAN_TABLE = ""
RUN_METRICS_TABLE = ""
BENCHMARK_EVENT_TABLE = ""
BASELINE_TABLE = ""

STAGING_ROOT = "/lakehouse/default/Files/executor-partition-benchmarks"
MODELS_DIR = ""
CAPACITY_SKU = "UNSET"
RUNTIME_VERSION = "UNSET"
BASELINE_BENCHMARK_BATCH_ID = ""
BASELINE_CONFIG_SHA256 = ""
MIN_WALL_SECONDS = 0
MIN_THROUGHPUT_IMPROVEMENT_PERCENT = 0.0
MAX_BENCHMARK_ITEMS = 1000
METADATA_RELATIVE_TOLERANCE = 0.02
INPUT_TXN_VERSION = 0
ENFORCE_GATE = True

PIPELINE = "rtdetr-osnet"
DEVICE_VARIANT = "cpu"
DEVICE = "cpu"
BATCH_SIZE = 1
SAMPLE_FPS = 3.0
DETECTION_THRESHOLD = 0.6
USE_FP16 = False
LINE = []
DETECTOR_MODEL = "r18"
CAMERA_MOTION_COMPENSATION = None

CPUS_PER_TASK = 1
DRIVER_MEMORY = "28g"
DRIVER_CORES = 4
EXECUTOR_MEMORY = "28g"
EXECUTOR_CORES = 4
MIN_EXECUTORS = 1
MAX_EXECUTORS = 4
PARALLEL_TASKS = "auto"
PARTITION_WAVES = 3
RUNTIME_AFFINITY_MAX_IMBALANCE = 0.10
PEAK_WORKER_MEMORY_GIB = 0.0
USABLE_EXECUTOR_MEMORY_GIB = 0.0
EXECUTOR_MEMORY_HEADROOM_FRACTION = 0.25


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit
import hashlib
import json
import math
import re
import time

from pyspark.sql import SparkSession, functions as F, types as T


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def parameter_bool(value, name):
    if isinstance(value, bool):
        return value
    if isinstance(value, str) and value.strip().lower() in {"true", "false"}:
        return value.strip().lower() == "true"
    raise ValueError(f"{name} must be true or false")


def optional_bool(value, name):
    if value is None or value == "":
        return None
    return parameter_bool(value, name)


def normalize_line(value):
    parsed = json.loads(value) if isinstance(value, str) else value
    if parsed in (None, []):
        return []
    if not isinstance(parsed, list) or len(parsed) != 4:
        raise ValueError("LINE must contain exactly four integer coordinates")
    if any(isinstance(item, bool) or int(item) != item for item in parsed):
        raise ValueError("LINE must contain exactly four integer coordinates")
    return [int(item) for item in parsed]


def positive_finite(value, name):
    if isinstance(value, bool):
        raise ValueError(f"{name} must be a positive finite number")
    parsed = float(value)
    if not math.isfinite(parsed) or parsed <= 0:
        raise ValueError(f"{name} must be a positive finite number")
    return parsed


def optional_positive_finite(value, name):
    if value is None or value == "":
        return None
    return positive_finite(value, name)


def optional_nonnegative_int(value, name):
    if value is None or value == "":
        return None
    if isinstance(value, bool):
        raise ValueError(f"{name} must be a non-negative integer")
    parsed = int(value)
    if parsed < 0 or float(parsed) != float(value):
        raise ValueError(f"{name} must be a non-negative integer")
    return parsed


def parse_benchmark_items(value, max_items):
    if type(max_items) is not int or max_items < 1:
        raise ValueError("MAX_BENCHMARK_ITEMS must be a positive integer")
    items = json.loads(value) if isinstance(value, str) else value
    if not isinstance(items, list) or not items:
        raise ValueError("BENCHMARK_ITEMS_JSON must contain a non-empty JSON array")
    if len(items) > max_items:
        raise ValueError(f"Benchmark contains {len(items)} items; maximum is {max_items}")
    parsed = []
    for index, item in enumerate(items):
        if not isinstance(item, dict):
            raise ValueError(f"Benchmark item {index} must be an object")
        video_uri = str(item.get("video_uri", "")).strip()
        sample_name = str(item.get("sample_name", "")).strip()
        if not video_uri or not sample_name:
            raise ValueError(
                f"Benchmark item {index} requires non-empty video_uri and sample_name"
            )
        parsed.append(
            {
                "ordinal": index,
                "video_uri": video_uri,
                "sample_name": sample_name,
                "duration_seconds": optional_positive_finite(
                    item.get("duration_seconds"), f"item {index} duration_seconds"
                ),
                "source_width": optional_positive_finite(
                    item.get("source_width"), f"item {index} source_width"
                ),
                "source_height": optional_positive_finite(
                    item.get("source_height"), f"item {index} source_height"
                ),
                "expected_line_in_count": optional_nonnegative_int(
                    item.get("expected_line_in_count"),
                    f"item {index} expected_line_in_count",
                ),
                "expected_line_out_count": optional_nonnegative_int(
                    item.get("expected_line_out_count"),
                    f"item {index} expected_line_out_count",
                ),
            }
        )
    return parsed


def relative_difference(left, right):
    return abs(float(left) - float(right)) / max(abs(float(left)), abs(float(right)), 1.0)


def resolve_media_metadata(item, probed, tolerance):
    if not 0 <= tolerance < 1:
        raise ValueError("METADATA_RELATIVE_TOLERANCE must be in [0, 1)")
    resolved = {}
    for name in ("duration_seconds", "source_width", "source_height"):
        supplied = item.get(name)
        observed = positive_finite(probed.get(name), f"probed {name}")
        if supplied is not None and relative_difference(supplied, observed) > tolerance:
            raise ValueError(
                f"{name} mismatch for {item['sample_name']}: "
                f"supplied={supplied}, probed={observed}, tolerance={tolerance}"
            )
        resolved[name] = float(supplied) if supplied is not None else observed
    resolved["source_fps"] = positive_finite(probed.get("source_fps"), "probed source_fps")
    resolved["source_frame_count"] = int(
        positive_finite(probed.get("source_frame_count"), "probed source_frame_count")
    )
    resolved["codec"] = str(probed.get("codec", "")).strip() or "UNKNOWN"
    return resolved


def stable_hash(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, separators=(",", ":")).encode("utf-8")
    ).hexdigest()


def deterministic_work_id(batch_id, ordinal, video_uri):
    return stable_hash(
        {"benchmark_batch_id": batch_id, "ordinal": ordinal, "video_uri": video_uri}
    )


def identifier(value, name, allow_empty=False):
    stripped = str(value).strip()
    if allow_empty and not stripped:
        return ""
    if IDENTIFIER.fullmatch(stripped) is None:
        raise ValueError(f"{name} is not a valid SQL identifier: {value!r}")
    return stripped


database = identifier(DATABASE, "DATABASE", allow_empty=True)
prefix = identifier(TABLE_PREFIX, "TABLE_PREFIX")
mode = str(MODE).strip().upper()
if mode not in {"PREPARE", "EVALUATE"}:
    raise ValueError("MODE must be PREPARE or EVALUATE")
benchmark_batch_id = str(BENCHMARK_BATCH_ID).strip()
if not benchmark_batch_id:
    raise ValueError("BENCHMARK_BATCH_ID is required")
capacity_sku = str(CAPACITY_SKU).strip()
runtime_version = str(RUNTIME_VERSION).strip()
for name, value in {
    "CAPACITY_SKU": capacity_sku,
    "RUNTIME_VERSION": runtime_version,
}.items():
    if not value or value.upper() == "UNSET":
        raise ValueError(f"{name} must be an exact non-placeholder value")

spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session with a default Lakehouse is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")


def table(suffix):
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


input_table = INPUT_TABLE or table("executor_partition_input")
output_table = OUTPUT_TABLE or table("executor_partition_records")
resource_snapshot_table = RESOURCE_SNAPSHOT_TABLE or table("executor_resource_snapshots")
partition_plan_table = PARTITION_PLAN_TABLE or table("executor_partition_plans")
run_metrics_table = RUN_METRICS_TABLE or table("executor_inference_runs")
benchmark_event_table = BENCHMARK_EVENT_TABLE or table("executor_benchmark_events")
baseline_table = BASELINE_TABLE or table("processing_benchmarks")

line_value = normalize_line(LINE)
camera_motion_compensation = optional_bool(
    CAMERA_MOTION_COMPENSATION, "CAMERA_MOTION_COMPENSATION"
)
config_value = {
    "pipeline": PIPELINE,
    "device_variant": DEVICE_VARIANT,
    "device": DEVICE,
    "batch_size": int(BATCH_SIZE),
    "sample_fps": float(SAMPLE_FPS),
    "detection_threshold": float(DETECTION_THRESHOLD),
    "use_fp16": parameter_bool(USE_FP16, "USE_FP16"),
    "line": line_value,
    "detector_model": DETECTOR_MODEL,
    "camera_motion_compensation": camera_motion_compensation,
    "models_dir": str(MODELS_DIR).strip(),
    "cpus_per_task": int(CPUS_PER_TASK),
    "driver_memory": str(DRIVER_MEMORY),
    "driver_cores": int(DRIVER_CORES),
    "executor_memory": str(EXECUTOR_MEMORY),
    "executor_cores": int(EXECUTOR_CORES),
    "min_executors": int(MIN_EXECUTORS),
    "max_executors": int(MAX_EXECUTORS),
    "parallel_tasks": PARALLEL_TASKS,
    "partition_waves": int(PARTITION_WAVES),
    "runtime_affinity_max_imbalance": float(RUNTIME_AFFINITY_MAX_IMBALANCE),
    "peak_worker_memory_gib": float(PEAK_WORKER_MEMORY_GIB),
    "usable_executor_memory_gib": float(USABLE_EXECUTOR_MEMORY_GIB),
    "executor_memory_headroom_fraction": float(EXECUTOR_MEMORY_HEADROOM_FRACTION),
}
config_sha256 = stable_hash(config_value)


def write_event(event_type, status, item_count, payload):
    event = {
        "event_id": stable_hash(
            {
                "benchmark_batch_id": benchmark_batch_id,
                "event_type": event_type,
                "run_id": str(RUN_ID).strip(),
            }
        ),
        "benchmark_batch_id": benchmark_batch_id,
        "run_id": str(RUN_ID).strip() or None,
        "event_type": event_type,
        "status": status,
        "capacity_sku": capacity_sku,
        "runtime_version": runtime_version,
        "config_sha256": config_sha256,
        "item_count": int(item_count),
        "payload_json": json.dumps(payload, sort_keys=True, separators=(",", ":")),
        "created_at_utc": datetime.now(timezone.utc),
    }
    event_schema = T.StructType(
        [
            T.StructField("event_id", T.StringType(), False),
            T.StructField("benchmark_batch_id", T.StringType(), False),
            T.StructField("run_id", T.StringType(), True),
            T.StructField("event_type", T.StringType(), False),
            T.StructField("status", T.StringType(), False),
            T.StructField("capacity_sku", T.StringType(), False),
            T.StructField("runtime_version", T.StringType(), False),
            T.StructField("config_sha256", T.StringType(), False),
            T.StructField("item_count", T.LongType(), False),
            T.StructField("payload_json", T.StringType(), False),
            T.StructField("created_at_utc", T.TimestampType(), False),
        ]
    )
    (
        spark_session.createDataFrame([event], schema=event_schema)
        .write.format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .option("txnAppId", f"executor-benchmark-event:{event['event_id']}")
        .option("txnVersion", 0)
        .saveAsTable(benchmark_event_table)
    )


In [ ]:
def prepare_benchmark():
    import cv2
    import notebookutils

    items = parse_benchmark_items(BENCHMARK_ITEMS_JSON, int(MAX_BENCHMARK_ITEMS))
    expected_item_count = int(EXPECTED_ITEM_COUNT)
    if expected_item_count not in (0, len(items)):
        raise ValueError(
            f"EXPECTED_ITEM_COUNT={expected_item_count} does not match submitted items={len(items)}"
        )
    if isinstance(INPUT_TXN_VERSION, bool):
        raise ValueError("INPUT_TXN_VERSION must be a non-negative integer")
    input_txn_version = int(INPUT_TXN_VERSION)
    if input_txn_version < 0 or float(input_txn_version) != float(INPUT_TXN_VERSION):
        raise ValueError("INPUT_TXN_VERSION must be a non-negative integer")
    models_dir = str(MODELS_DIR).strip()
    if not models_dir:
        raise ValueError("MODELS_DIR is required")
    if not notebookutils.fs.exists(models_dir):
        raise FileNotFoundError(f"MODELS_DIR is not accessible: {models_dir}")

    batch_segment = stable_hash({"benchmark_batch_id": benchmark_batch_id})[:24]
    staging_dir = f"{str(STAGING_ROOT).rstrip('/')}/{batch_segment}/inputs"
    notebookutils.fs.mkdirs(staging_dir)
    prepare_started = time.monotonic()
    prepared_at = datetime.now(timezone.utc)
    staged = {}

    def probe_video(path):
        capture = cv2.VideoCapture(path)
        if not capture.isOpened():
            capture.release()
            raise ValueError(f"Could not open staged benchmark video: {path}")
        try:
            fps = float(capture.get(cv2.CAP_PROP_FPS))
            frames = float(capture.get(cv2.CAP_PROP_FRAME_COUNT))
            width = float(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = float(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
            codec_value = int(capture.get(cv2.CAP_PROP_FOURCC))
        finally:
            capture.release()
        if fps <= 0 or frames <= 0 or width <= 0 or height <= 0:
            raise ValueError(f"Invalid media metadata for staged video: {path}")
        codec = "".join(chr((codec_value >> (8 * index)) & 0xFF) for index in range(4))
        return {
            "duration_seconds": frames / fps,
            "source_width": width,
            "source_height": height,
            "source_fps": fps,
            "source_frame_count": frames,
            "codec": codec.strip() or "UNKNOWN",
        }

    for item in items:
        video_uri = item["video_uri"]
        if video_uri in staged:
            continue
        if not notebookutils.fs.exists(video_uri):
            raise FileNotFoundError(f"Benchmark video is not accessible: {video_uri}")
        suffix = Path(urlsplit(video_uri).path).suffix or ".video"
        destination = f"{staging_dir}/{stable_hash({'video_uri': video_uri})[:24]}{suffix}"
        if not notebookutils.fs.exists(destination):
            notebookutils.fs.cp(video_uri, destination, False)
        staged[video_uri] = {
            "local_video_path": destination,
            "probed": probe_video(destination),
        }

    rows = []
    for item in items:
        stage = staged[item["video_uri"]]
        metadata = resolve_media_metadata(
            item,
            stage["probed"],
            float(METADATA_RELATIVE_TOLERANCE),
        )
        rows.append(
            {
                "benchmark_batch_id": benchmark_batch_id,
                "capacity_sku": capacity_sku,
                "runtime_version": runtime_version,
                "config_sha256": config_sha256,
                "item_ordinal": int(item["ordinal"]),
                "sample_name": item["sample_name"],
                "source_uri": item["video_uri"],
                "work_id": deterministic_work_id(
                    benchmark_batch_id, item["ordinal"], item["video_uri"]
                ),
                "attempt_id": None,
                "local_video_path": stage["local_video_path"],
                "pipeline": PIPELINE,
                "device_variant": DEVICE_VARIANT,
                "device": DEVICE,
                "batch_size": int(BATCH_SIZE),
                "sample_fps": float(SAMPLE_FPS),
                "detection_threshold": float(DETECTION_THRESHOLD),
                "use_fp16": parameter_bool(USE_FP16, "USE_FP16"),
                "line": line_value or None,
                "detector_model": str(DETECTOR_MODEL),
                "camera_motion_compensation": camera_motion_compensation,
                "models_dir": models_dir,
                "duration_seconds": float(metadata["duration_seconds"]),
                "source_width": int(metadata["source_width"]),
                "source_height": int(metadata["source_height"]),
                "source_fps": float(metadata["source_fps"]),
                "source_frame_count": int(metadata["source_frame_count"]),
                "codec": metadata["codec"],
                "expected_line_in_count": item["expected_line_in_count"],
                "expected_line_out_count": item["expected_line_out_count"],
                "prepared_at_utc": prepared_at,
            }
        )

    preflight_schema = T.StructType(
        [
            T.StructField("local_video_path", T.StringType(), False),
            T.StructField("models_dir", T.StringType(), False),
        ]
    )

    def check_executor_paths(partition):
        for row in partition:
            video_path = str(row["local_video_path"])
            model_path = str(row["models_dir"])
            yield (
                video_path,
                model_path,
                Path(video_path).is_file(),
                Path(model_path).is_dir(),
            )

    unique_paths = sorted({(row["local_video_path"], row["models_dir"]) for row in rows})
    preflight = (
        spark_session.createDataFrame(unique_paths, schema=preflight_schema)
        .repartition(min(len(unique_paths), max(1, spark_session.sparkContext.defaultParallelism)))
        .rdd.mapPartitions(check_executor_paths)
        .collect()
    )
    inaccessible = [
        {
            "local_video_path": video_path,
            "models_dir": model_path,
            "video_accessible": video_accessible,
            "models_accessible": models_accessible,
        }
        for video_path, model_path, video_accessible, models_accessible in preflight
        if not video_accessible or not models_accessible
    ]
    if inaccessible:
        raise RuntimeError(f"Executor path preflight failed: {inaccessible}")

    input_schema = T.StructType(
        [
            T.StructField("benchmark_batch_id", T.StringType(), False),
            T.StructField("capacity_sku", T.StringType(), False),
            T.StructField("runtime_version", T.StringType(), False),
            T.StructField("config_sha256", T.StringType(), False),
            T.StructField("item_ordinal", T.IntegerType(), False),
            T.StructField("sample_name", T.StringType(), False),
            T.StructField("source_uri", T.StringType(), False),
            T.StructField("work_id", T.StringType(), False),
            T.StructField("attempt_id", T.StringType(), True),
            T.StructField("local_video_path", T.StringType(), False),
            T.StructField("pipeline", T.StringType(), False),
            T.StructField("device_variant", T.StringType(), False),
            T.StructField("device", T.StringType(), False),
            T.StructField("batch_size", T.IntegerType(), False),
            T.StructField("sample_fps", T.DoubleType(), False),
            T.StructField("detection_threshold", T.DoubleType(), False),
            T.StructField("use_fp16", T.BooleanType(), False),
            T.StructField("line", T.ArrayType(T.IntegerType()), True),
            T.StructField("detector_model", T.StringType(), False),
            T.StructField("camera_motion_compensation", T.BooleanType(), True),
            T.StructField("models_dir", T.StringType(), False),
            T.StructField("duration_seconds", T.DoubleType(), False),
            T.StructField("source_width", T.IntegerType(), False),
            T.StructField("source_height", T.IntegerType(), False),
            T.StructField("source_fps", T.DoubleType(), False),
            T.StructField("source_frame_count", T.LongType(), False),
            T.StructField("codec", T.StringType(), False),
            T.StructField("expected_line_in_count", T.LongType(), True),
            T.StructField("expected_line_out_count", T.LongType(), True),
            T.StructField("prepared_at_utc", T.TimestampType(), False),
        ]
    )
    frame = spark_session.createDataFrame(rows, schema=input_schema)
    if spark_session.catalog.tableExists(input_table):
        existing = spark_session.table(input_table).where(
            F.col("benchmark_batch_id") == benchmark_batch_id
        )
        existing_rows = existing.select("work_id", "config_sha256").collect()
        if existing_rows:
            existing_ids = {row["work_id"] for row in existing_rows}
            expected_ids = {row["work_id"] for row in rows}
            existing_hashes = {row["config_sha256"] for row in existing_rows}
            if existing_ids != expected_ids or existing_hashes != {config_sha256}:
                raise RuntimeError(
                    "BENCHMARK_BATCH_ID already contains different prepared input; "
                    "use a new batch ID"
                )
        else:
            (
                frame.write.format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .option("txnAppId", f"executor-benchmark-input:{benchmark_batch_id}")
                .option("txnVersion", input_txn_version)
                .saveAsTable(input_table)
            )
    else:
        (
            frame.write.format("delta")
            .mode("append")
            .option("txnAppId", f"executor-benchmark-input:{benchmark_batch_id}")
            .option("txnVersion", input_txn_version)
            .saveAsTable(input_table)
        )

    outcome = {
        "mode": "PREPARE",
        "benchmark_batch_id": benchmark_batch_id,
        "config_sha256": config_sha256,
        "prepared_item_count": len(rows),
        "distinct_source_count": len(staged),
        "input_table": input_table,
        "prepare_seconds": time.monotonic() - prepare_started,
    }
    write_event("PREPARED", "SUCCEEDED", len(rows), outcome)
    return outcome


In [ ]:
def evaluate_benchmark():
    run_id = str(RUN_ID).strip()
    if not run_id:
        raise ValueError("RUN_ID is required in EVALUATE mode")
    expected_count = int(EXPECTED_ITEM_COUNT)
    if expected_count < 1:
        raise ValueError("EXPECTED_ITEM_COUNT must be positive in EVALUATE mode")
    min_wall_seconds = float(MIN_WALL_SECONDS)
    if min_wall_seconds < 0:
        raise ValueError("MIN_WALL_SECONDS cannot be negative")

    prepared = spark_session.table(input_table).where(
        (F.col("benchmark_batch_id") == benchmark_batch_id)
        & (F.col("capacity_sku") == capacity_sku)
        & (F.col("runtime_version") == runtime_version)
        & (F.col("config_sha256") == config_sha256)
    )
    records = spark_session.table(output_table).where(
        (F.col("run_id") == run_id)
        & (F.col("benchmark_batch_id") == benchmark_batch_id)
        & (F.col("capacity_sku") == capacity_sku)
        & (F.col("runtime_version") == runtime_version)
        & (F.col("config_sha256") == config_sha256)
    )
    plans = spark_session.table(partition_plan_table).where(
        (F.col("run_id") == run_id)
        & (F.col("benchmark_batch_id") == benchmark_batch_id)
        & (F.col("capacity_sku") == capacity_sku)
        & (F.col("runtime_version") == runtime_version)
        & (F.col("config_sha256") == config_sha256)
    )
    resources = spark_session.table(resource_snapshot_table).where(
        (F.col("run_id") == run_id)
        & (F.col("benchmark_batch_id") == benchmark_batch_id)
        & (F.col("capacity_sku") == capacity_sku)
        & (F.col("runtime_version") == runtime_version)
        & (F.col("config_sha256") == config_sha256)
    )
    metrics_rows = spark_session.table(run_metrics_table).where(
        (F.col("run_id") == run_id)
        & (F.col("benchmark_batch_id") == benchmark_batch_id)
        & (F.col("capacity_sku") == capacity_sku)
        & (F.col("runtime_version") == runtime_version)
        & (F.col("config_sha256") == config_sha256)
    ).collect()

    failures = []
    prepared_count = prepared.count()
    if prepared_count != expected_count:
        failures.append(
            f"prepared item count mismatch: expected={expected_count}, actual={prepared_count}"
        )
    if len(metrics_rows) != 1:
        failures.append(f"expected exactly one run metric row, found {len(metrics_rows)}")
        metric = None
    else:
        metric = metrics_rows[0]

    terminal = records.where(F.col("record_type").isin("video_result", "error"))
    terminal_summary = terminal.agg(
        F.count("*").alias("terminal_count"),
        F.countDistinct("work_id").alias("distinct_work_count"),
        F.sum((F.col("status") == "FAILED").cast("long")).alias("failed_count"),
    ).first()
    terminal_count = int(terminal_summary["terminal_count"] or 0)
    distinct_work_count = int(terminal_summary["distinct_work_count"] or 0)
    failed_count = int(terminal_summary["failed_count"] or 0)
    if terminal_count != expected_count or distinct_work_count != expected_count:
        failures.append(
            "terminal result completeness failed: "
            f"rows={terminal_count}, distinct_work={distinct_work_count}, expected={expected_count}"
        )
    if failed_count:
        failures.append(f"{failed_count} videos failed")
    plan_count = plans.count()
    if plan_count != expected_count or plans.select("work_id").distinct().count() != expected_count:
        failures.append("partition plan does not contain every prepared work item")

    task_summary = records.agg(
        F.countDistinct("task_cpus").alias("task_cpu_values"),
        F.min("task_cpus").alias("task_cpus"),
        F.countDistinct("native_threads").alias("native_thread_values"),
        F.min("native_threads").alias("native_threads"),
    ).first()
    if (
        int(task_summary["task_cpu_values"] or 0) != 1
        or int(task_summary["task_cpus"] or 0) != int(CPUS_PER_TASK)
        or int(task_summary["native_thread_values"] or 0) != 1
        or not 1 <= int(task_summary["native_threads"] or 0) <= int(CPUS_PER_TASK)
    ):
        failures.append("task CPU or native-thread allocations conflict with the request")

    resource_summary = resources.agg(
        F.count("*").alias("snapshot_count"),
        F.sum((F.col("phase") == "before").cast("long")).alias("before_count"),
        F.sum((F.col("phase") == "after").cast("long")).alias("after_count"),
        F.sum((~F.col("complete")).cast("long")).alias("incomplete_count"),
        F.sum(F.col("fallback_used").cast("long")).alias("fallback_count"),
        F.countDistinct("discovery_source").alias("discovery_source_values"),
        F.min("discovery_source").alias("discovery_source"),
        F.countDistinct("effective_task_cpus").alias("task_cpu_values"),
        F.min("effective_task_cpus").alias("effective_task_cpus"),
    ).first()
    if int(resource_summary["snapshot_count"] or 0) != 2:
        failures.append("expected complete before and after resource snapshots")
    if (
        int(resource_summary["before_count"] or 0) != 1
        or int(resource_summary["after_count"] or 0) != 1
    ):
        failures.append("resource snapshots do not contain one before and one after phase")
    if int(resource_summary["incomplete_count"] or 0):
        failures.append("resource discovery was incomplete")
    if int(resource_summary["fallback_count"] or 0):
        failures.append("resource discovery used assumed executors")
    if (
        int(resource_summary["discovery_source_values"] or 0) != 1
        or resource_summary["discovery_source"] != "spark-monitoring-rest-v1"
    ):
        failures.append("resource snapshots do not use the supported monitoring backend")
    if (
        int(resource_summary["task_cpu_values"] or 0) != 1
        or int(resource_summary["effective_task_cpus"] or 0) != int(CPUS_PER_TASK)
    ):
        failures.append("requested and effective task CPU allocations do not match")

    if metric is not None:
        if not bool(metric["metrics_complete"]):
            failures.append("run metrics are incomplete")
        if int(metric["resumed_batch_count"]) != 0:
            failures.append("run resumed previously committed batches")
        if int(metric["assumed_duration_count"]) != 0:
            failures.append("run used assumed source durations")
        if int(metric["memory_safe_workers_per_executor"]) < 1:
            failures.append("run did not retain a memory-safe worker slot per executor")
        if int(metric["video_count"]) != expected_count:
            failures.append("run metric video count does not match submitted items")
        if int(metric["succeeded_videos"]) != expected_count or int(metric["failed_videos"]):
            failures.append("run metric terminal counts do not show complete success")
        if float(metric["wall_seconds"]) < min_wall_seconds:
            failures.append(
                f"run wall time {metric['wall_seconds']}s is below {min_wall_seconds}s"
            )
        throughput = metric["throughput_video_minutes_per_wall_minute"]
        if throughput is None or float(throughput) <= 0:
            failures.append("successful source-video throughput is unavailable")
        observed_concurrency = metric["observed_concurrency"]
        if (
            observed_concurrency is None
            or int(observed_concurrency) > int(metric["planned_concurrency"])
        ):
            failures.append("observed concurrency is missing or exceeds the plan")
    else:
        throughput = None

    successful_results = records.where(
        (F.col("record_type") == "video_result")
        & (F.col("status") == "SUCCEEDED")
    ).select(
        "work_id",
        F.get_json_object("payload_json", "$.line_in_count").cast("long").alias("line_in"),
        F.get_json_object("payload_json", "$.line_out_count").cast("long").alias("line_out"),
    )
    line_mismatches = (
        prepared.select(
            "work_id", "expected_line_in_count", "expected_line_out_count"
        )
        .join(successful_results, on="work_id", how="left")
        .where(
            (
                F.col("expected_line_in_count").isNotNull()
                & (
                    F.col("line_in").isNull()
                    | (F.col("expected_line_in_count") != F.col("line_in"))
                )
            )
            | (
                F.col("expected_line_out_count").isNotNull()
                & (
                    F.col("line_out").isNull()
                    | (F.col("expected_line_out_count") != F.col("line_out"))
                )
            )
        )
        .count()
    )
    if line_mismatches:
        failures.append(f"{line_mismatches} videos failed expected line-count checks")

    baseline_speed_x = None
    improvement_percent = None
    baseline_batch_id = str(BASELINE_BENCHMARK_BATCH_ID).strip()
    if baseline_batch_id:
        baseline_config = str(BASELINE_CONFIG_SHA256).strip()
        if not baseline_config:
            failures.append(
                "BASELINE_CONFIG_SHA256 is required with BASELINE_BENCHMARK_BATCH_ID"
            )
        baseline = spark_session.table(baseline_table).where(
            F.col("benchmark_batch_id") == baseline_batch_id
        )
        if baseline_config:
            baseline = baseline.where(F.col("config_sha256") == baseline_config)
        prepared_samples = prepared.groupBy("sample_name").count().withColumnRenamed(
            "count", "prepared_count"
        )
        baseline_samples = baseline.groupBy("sample_name").count().withColumnRenamed(
            "count", "baseline_count"
        )
        sample_mismatches = (
            prepared_samples.join(baseline_samples, on="sample_name", how="full")
            .where(
                F.coalesce(F.col("prepared_count"), F.lit(0))
                != F.coalesce(F.col("baseline_count"), F.lit(0))
            )
            .count()
        )
        if sample_mismatches:
            failures.append("baseline sample multiset does not match prepared input")
        baseline_summary = baseline.agg(
            F.count("*").alias("row_count"),
            F.sum((~F.col("succeeded")).cast("long")).alias("failed_count"),
            F.countDistinct("capacity_sku").alias("capacity_values"),
            F.countDistinct("runtime_version").alias("runtime_values"),
            F.min("capacity_sku").alias("capacity_sku"),
            F.min("runtime_version").alias("runtime_version"),
            F.sum(F.when(F.col("succeeded"), F.col("video_duration_seconds")).otherwise(0.0)).alias("source_seconds"),
            F.min("benchmark_started_at").alias("started_at"),
            F.max("completed_at").alias("completed_at"),
        ).first()
        baseline_count = int(baseline_summary["row_count"] or 0)
        if baseline_count != expected_count:
            failures.append(
                f"baseline count mismatch: expected={expected_count}, actual={baseline_count}"
            )
        if int(baseline_summary["failed_count"] or 0):
            failures.append("baseline contains failed videos")
        if (
            int(baseline_summary["capacity_values"] or 0) != 1
            or baseline_summary["capacity_sku"] != capacity_sku
        ):
            failures.append("baseline capacity label does not match")
        if (
            int(baseline_summary["runtime_values"] or 0) != 1
            or baseline_summary["runtime_version"] != runtime_version
        ):
            failures.append("baseline runtime label does not match")
        if baseline_summary["started_at"] and baseline_summary["completed_at"]:
            baseline_wall_seconds = (
                baseline_summary["completed_at"] - baseline_summary["started_at"]
            ).total_seconds()
            if baseline_wall_seconds > 0:
                baseline_speed_x = (
                    float(baseline_summary["source_seconds"]) / baseline_wall_seconds
                )
        if baseline_speed_x is None or baseline_speed_x <= 0:
            failures.append("baseline aggregate throughput is unavailable")
        elif throughput is not None:
            executor_speed_x = float(throughput)
            improvement_percent = 100.0 * (executor_speed_x / baseline_speed_x - 1.0)
            if improvement_percent < float(MIN_THROUGHPUT_IMPROVEMENT_PERCENT):
                failures.append(
                    "throughput improvement gate failed: "
                    f"actual={improvement_percent}, "
                    f"required={MIN_THROUGHPUT_IMPROVEMENT_PERCENT}"
                )

    outcome = {
        "mode": "EVALUATE",
        "benchmark_batch_id": benchmark_batch_id,
        "run_id": run_id,
        "config_sha256": config_sha256,
        "expected_item_count": expected_count,
        "prepared_item_count": prepared_count,
        "terminal_result_count": terminal_count,
        "failed_video_count": failed_count,
        "line_count_mismatches": line_mismatches,
        "throughput_video_minutes_per_wall_minute": (
            float(throughput) if throughput is not None else None
        ),
        "baseline_speed_x": baseline_speed_x,
        "improvement_percent": improvement_percent,
        "gate_passed": not failures,
        "failures": failures,
    }
    write_event(
        "EVALUATED",
        "SUCCEEDED" if not failures else "FAILED",
        expected_count,
        outcome,
    )
    if failures and parameter_bool(ENFORCE_GATE, "ENFORCE_GATE"):
        raise RuntimeError("Executor benchmark gate failed: " + "; ".join(failures))
    return outcome


In [ ]:
outcome = prepare_benchmark() if mode == "PREPARE" else evaluate_benchmark()
print(json.dumps(outcome, sort_keys=True, default=str))

try:
    import notebookutils

    notebookutils.notebook.exit(json.dumps(outcome, sort_keys=True, default=str))
except ImportError:
    pass
